# DATA ANALYSIS 
## DATA COVERAGE 
### 1. DATA CLEANING


 #### c. IMF Climate Change

#### Data Scraping 

In [ ]:
 # ============================================================
# 1. Import packages
# ============================================================

import os
import re
import io
import sys
import time
import zipfile
import subprocess
import unicodedata
import requests
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
 # ============================================================
# 2. Paths and settings
# ============================================================

# IMF / ArcGIS catalog item for the indicators summary
catalog_item_id = "74a9db2b716d46f2b25c2ec53c0ed1ce"

# Output folders
raw_download_folder = "imf_climate_raw_downloads"
dataset_folder = "imf_climate_country_coverage_datasets"

os.makedirs(raw_download_folder, exist_ok=True)
os.makedirs(dataset_folder, exist_ok=True)

# Output files
indicators_summary_file = "00_Indicators_Summary.csv"

manifest_file = os.path.join(
    dataset_folder,
    "imf_climate_country_coverage_manifest.csv"
)

# Request settings
request_timeout = 180
pause_seconds = 0.25

# Browser-like request headers
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/124.0 Safari/537.36"
    ),
    "Accept": "text/csv,application/csv,application/json,text/plain,*/*",
    "Referer": "https://climatedata.imf.org/",
    "Origin": "https://climatedata.imf.org"
}

print("Scraping paths and settings are ready.")
print("Raw download folder:", raw_download_folder)
print("Dataset folder:", dataset_folder)
print("Manifest file:", manifest_file)

In [ ]:
# ============================================================
# 3. Helper functions for scraping
# ============================================================

def clean_text(value):
    """Clean text values by removing extra spaces."""
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def slugify(value, max_len=140):
    """
    Convert indicator names into safe file names.
    Example:
        'Carbon Tax to Revenues/Assets' -> 'Carbon_Tax_to_Revenues_Assets'
    """
    text_value = unicodedata.normalize("NFKD", str(value))
    text_value = text_value.encode("ascii", "ignore").decode("ascii")
    text_value = re.sub(r"[^A-Za-z0-9]+", "_", text_value)
    text_value = re.sub(r"_+", "_", text_value).strip("_")

    if text_value == "":
        text_value = "dataset"

    return text_value[:max_len]


def normalize_columns(df):
    """
    Clean column names and remove unnamed columns.
    """
    df = df.copy()

    df.columns = [
        clean_text(col)
        for col in df.columns
    ]

    unnamed_cols = [
        col
        for col in df.columns
        if str(col).lower().startswith("unnamed")
    ]

    if len(unnamed_cols) > 0:
        df = df.drop(columns=unnamed_cols)

    return df


def extract_item_id_from_url(url_value):
    """
    Extract ArcGIS item ID from IMF download URLs.
    """
    url_text = clean_text(url_value)

    match = re.search(r"/datasets/([A-Za-z0-9]+)/", url_text)
    if match is not None:
        return match.group(1)

    match = re.search(r"/items/([A-Za-z0-9]+)", url_text)
    if match is not None:
        return match.group(1)

    return ""


def get_url_content(url_value):
    """
    Download content from a URL.
    """
    response = requests.get(
        url_value,
        headers=headers,
        timeout=request_timeout
    )

    response.raise_for_status()
    return response.content


def read_csv_from_response_bytes(content_bytes):
    """
    Read CSV content from either direct CSV bytes or ZIP bytes.
    Some ArcGIS endpoints return ZIP files containing CSV files.
    """
    # ZIP files usually begin with b"PK"
    if content_bytes[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(content_bytes)) as zip_file:
            csv_names = [
                name
                for name in zip_file.namelist()
                if name.lower().endswith(".csv")
            ]

            if len(csv_names) == 0:
                raise ValueError("ZIP response did not contain any CSV files.")

            csv_names = sorted(csv_names)

            with zip_file.open(csv_names[0]) as file_obj:
                df = pd.read_csv(file_obj, low_memory=False)

            return normalize_columns(df)

    # Direct CSV response
    df = pd.read_csv(io.BytesIO(content_bytes), low_memory=False)
    return normalize_columns(df)


def try_read_csv_from_urls(urls):
    """
    Try several possible download URLs.
    Return the first successful dataframe, URL, and raw content bytes.
    """
    last_error = None

    for url_value in urls:
        try:
            content_bytes = get_url_content(url_value)
            df = read_csv_from_response_bytes(content_bytes)
            return df, url_value, content_bytes

        except Exception as error:
            last_error = error
            time.sleep(pause_seconds)

    raise RuntimeError(
        "All CSV download attempts failed. Last error: "
        + str(last_error)
    )


print("Scraping helper functions are ready.")

In [ ]:
 # ============================================================
# 4. Download IMF indicators summary catalog
# ============================================================

catalog_download_urls = [
    (
        "https://climatedata.imf.org/api/download/v1/items/"
        + catalog_item_id
        + "/csv?layers=0"
    ),
    (
        "https://climatedata-imf.opendata.arcgis.com/api/download/v1/items/"
        + catalog_item_id
        + "/csv?layers=0"
    ),
    (
        "https://opendata.arcgis.com/api/v3/datasets/"
        + catalog_item_id
        + "_0/downloads/data?format=csv&spatialRefId=4326&where=1%3D1"
    ),
    (
        "https://www.arcgis.com/sharing/rest/content/items/"
        + catalog_item_id
        + "/data"
    )
]

indicators_df, catalog_success_url, catalog_content_bytes = try_read_csv_from_urls(
    catalog_download_urls
)

indicators_df = normalize_columns(indicators_df)

indicators_df.to_csv(
    indicators_summary_file,
    index=False
)

print("Indicators summary downloaded successfully.")
print("Successful URL:")
print(catalog_success_url)

print("Indicators summary shape:")
print(indicators_df.shape)

print("Saved file:")
print(indicators_summary_file)

print("Preview:")
print(indicators_df.head())

In [ ]:
 # ============================================================
# 5. Check required catalog columns
# ============================================================

required_catalog_columns = [
    "Indicator_ID",
    "Indicator",
    "Country_Coverage",
    "Download_Data_URL"
]

missing_columns = [
    col
    for col in required_catalog_columns
    if col not in indicators_df.columns
]

if len(missing_columns) > 0:
    raise ValueError(
        "The indicators summary is missing required columns: "
        + ", ".join(missing_columns)
    )

print("Required catalog columns are available.")

In [ ]:
 # ============================================================
# 6. Filter catalog to country-level IMF Climate datasets
# ============================================================

# Keep country-level datasets.
# Remove datasets marked as Regional or Global.
selected_indicators_df = indicators_df[
    ~indicators_df["Country_Coverage"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin([
        "regional",
        "global"
    ])
].copy()

selected_indicators_df = selected_indicators_df.reset_index(drop=True)

print("Total indicators in catalog:")
print(len(indicators_df))

print("Selected country-level indicators:")
print(len(selected_indicators_df))

print("Preview of selected datasets:")
print(
    selected_indicators_df[
        [
            "Indicator_ID",
            "Indicator",
            "Country_Coverage",
            "Download_Data_URL"
        ]
    ].head()
)

In [ ]:
 # ============================================================
# 7. Download each selected country-level dataset
# ============================================================

download_manifest_rows = []
downloaded_file_paths = []

for row_index, row in tqdm(
    selected_indicators_df.iterrows(),
    total=len(selected_indicators_df),
    desc="Downloading IMF Climate country-level datasets"
):
    indicator_id = int(row["Indicator_ID"])
    indicator_name = clean_text(row["Indicator"])
    country_coverage = clean_text(row.get("Country_Coverage", ""))
    time_coverage = clean_text(row.get("Time_Coverage", ""))
    download_data_url = clean_text(row.get("Download_Data_URL", ""))

    item_id = extract_item_id_from_url(download_data_url)

    if item_id == "":
        download_manifest_rows.append({
            "Indicator_ID": indicator_id,
            "Indicator": indicator_name,
            "Country_Coverage": country_coverage,
            "Time_Coverage": time_coverage,
            "Download_Data_URL": download_data_url,
            "ArcGIS_Item_ID": "",
            "Downloaded_File": "",
            "Download_Status": "failed_no_item_id",
            "Successful_URL": ""
        })
        continue

    dataset_download_urls = [
        (
            "https://climatedata.imf.org/api/download/v1/items/"
            + item_id
            + "/csv?layers=0"
        ),
        (
            "https://climatedata-imf.opendata.arcgis.com/api/download/v1/items/"
            + item_id
            + "/csv?layers=0"
        ),
        (
            "https://opendata.arcgis.com/api/v3/datasets/"
            + item_id
            + "_0/downloads/data?format=csv&spatialRefId=4326&where=1%3D1"
        ),
        (
            "https://www.arcgis.com/sharing/rest/content/items/"
            + item_id
            + "/data"
        )
    ]

    try:
        dataset_df, success_url, content_bytes = try_read_csv_from_urls(
            dataset_download_urls
        )

        dataset_df = normalize_columns(dataset_df)

        file_name = (
            str(indicator_id).zfill(2)
            + "_"
            + slugify(indicator_name)
            + ".csv"
        )

        dataset_file_path = os.path.join(dataset_folder, file_name)
        raw_file_path = os.path.join(raw_download_folder, file_name)

        # Save cleaned CSV
        dataset_df.to_csv(
            dataset_file_path,
            index=False
        )

        # Save a raw-equivalent CSV copy for backup
        dataset_df.to_csv(
            raw_file_path,
            index=False
        )

        downloaded_file_paths.append(dataset_file_path)

        download_manifest_rows.append({
            "Indicator_ID": indicator_id,
            "Indicator": indicator_name,
            "Country_Coverage": country_coverage,
            "Time_Coverage": time_coverage,
            "Download_Data_URL": download_data_url,
            "ArcGIS_Item_ID": item_id,
            "Downloaded_File": file_name,
            "Download_Status": "success",
            "Successful_URL": success_url
        })

    except Exception as error:
        download_manifest_rows.append({
            "Indicator_ID": indicator_id,
            "Indicator": indicator_name,
            "Country_Coverage": country_coverage,
            "Time_Coverage": time_coverage,
            "Download_Data_URL": download_data_url,
            "ArcGIS_Item_ID": item_id,
            "Downloaded_File": "",
            "Download_Status": "failed_" + str(error)[:250],
            "Successful_URL": ""
        })

    time.sleep(pause_seconds)


manifest_df = pd.DataFrame(download_manifest_rows)

manifest_df.to_csv(
    manifest_file,
    index=False
)

print("Download process completed.")
print("Manifest shape:")
print(manifest_df.shape)

print("Successful downloads:")
print((manifest_df["Download_Status"] == "success").sum())

print("Failed downloads:")
print((manifest_df["Download_Status"] != "success").sum())

print("Manifest file saved:")
print(manifest_file)

print("Manifest preview:")
print(manifest_df.head())

In [ ]:
 # ============================================================
# 8. Final scraping validation
# ============================================================

downloaded_csv_files = sorted([
    file_name
    for file_name in os.listdir(dataset_folder)
    if file_name.lower().endswith(".csv")
    and file_name != os.path.basename(manifest_file)
])

print("Scraping validation summary")
print("---------------------------")

print("Indicators summary file exists:")
print(os.path.exists(indicators_summary_file))

print("Manifest file exists:")
print(os.path.exists(manifest_file))

print("Number of downloaded dataset CSV files:")
print(len(downloaded_csv_files))

print("First few downloaded files:")
print(downloaded_csv_files[:10])

print("Dataset folder:")
print(dataset_folder)

print("Raw backup folder:")
print(raw_download_folder)

print("Scraping script finished successfully.")

 ##### Data Cleaning

In [ ]:
 # ============================================================
# IMF Climate Data Cleaning Script
# Purpose:
#   1. Read downloaded IMF Climate country-level datasets.
#   2. Match reporting countries to template country columns.
#   3. Split datasets into meaningful indicator variants.
#   4. Code country availability as 1 or 0.
#   5. Save final CSV and Excel outputs.
#
# Required input:
#   IMF_final.csv
#   imf_climate_country_coverage_datasets/
#
# Optional input:
#   imf_climate_country_coverage_manifest.csv
# ============================================================

In [2]:
 # ============================================================
# 1. Import packages
# ============================================================

import os
import re
import subprocess
import sys
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Install country_converter if needed
try:
    import country_converter as coco
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "country_converter"
    ])
    import country_converter as coco

In [3]:
 # ============================================================
# 2. Paths and settings
# ============================================================

# Input paths
template_file = "IMF_final.csv"
data_folder = "imf_climate_country_coverage_datasets"

manifest_file = os.path.join(
    data_folder,
    "imf_climate_country_coverage_manifest.csv"
)

# Output paths
main_output_csv = "IMF_final_variant_country_availability.csv"

detail_output_csv = (
    "IMF_final_variant_country_availability_detail.csv"
)

rules_output_csv = (
    "IMF_final_variant_split_rules.csv"
)

excel_output_file = (
    "IMF_final_variant_country_availability.xlsx"
)

# Year handling
current_year = 2026
latest_year_min_share = 0.25
year_pattern = re.compile(r"^\d{4}$")

print("Data cleaning paths and settings are ready.")
print("Template file:", template_file)
print("Data folder:", data_folder)
print("Manifest file:", manifest_file)

Data cleaning paths and settings are ready.
Template file: IMF_final.csv
Data folder: imf_climate_country_coverage_datasets
Manifest file: imf_climate_country_coverage_datasets\imf_climate_country_coverage_manifest.csv


In [4]:
# ============================================================
# 3. Helper functions
# ============================================================

def clean_text(value):
    """
    Convert a value to clean text.
    """
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value).strip()
    )


def normalize_columns(df):
    """
    Clean column names and remove unnamed columns.

    This keeps IMF year columns such as F2019, F2020, and F2021 unchanged.
    """
    df = df.copy()

    df.columns = [
        clean_text(col)
        for col in df.columns
    ]

    unnamed_cols = [
        col
        for col in df.columns
        if str(col).lower().startswith("unnamed")
    ]

    if len(unnamed_cols) > 0:
        df = df.drop(columns=unnamed_cols)

    return df


def first_nonempty(series):
    """
    Return the first non-empty value in a pandas Series.
    """
    values = [
        clean_text(value)
        for value in series
        if clean_text(value) != ""
    ]

    if len(values) == 0:
        return ""

    return values[0]


def unique_join(series):
    """
    Join unique non-empty text values using semicolon.
    """
    seen = []

    for value in series:
        text = clean_text(value)

        if text != "" and text not in seen:
            seen.append(text)

    return "; ".join(seen)


def is_year_column(column_name):
    """
    Identify year columns.

    Handles both formats:
      2019
      F2019
    """
    column_text = str(column_name).strip()

    return re.match(r"^F?\d{4}$", column_text) is not None


def extract_year_from_column(column_name):
    """
    Extract the numeric year from a year column.

    Examples:
      2019  -> 2019
      F2019 -> 2019
    """
    column_text = str(column_name).strip()

    match = re.search(r"(\d{4})$", column_text)

    if match is None:
        return None

    return int(match.group(1))


def latest_reporting_years(df, min_share=0.25):
    """
    Select the latest two reporting year columns based on unique country coverage.

    This version handles IMF year columns written as:
      2019
      F2019

    It returns the actual dataframe column names, such as:
      ["F2021", "F2022"]

    This is important because the later code must use the real column names
    when checking whether each country has data.
    """

    year_columns = [
        column
        for column in df.columns
        if is_year_column(column)
    ]

    if len(year_columns) == 0:
        return []

    work_df = df.copy()

    # Create country identifier for coverage counting.
    if "ISO3" in work_df.columns:
        work_df["_country_id_for_year_check"] = (
            work_df["ISO3"]
            .astype(str)
            .str.strip()
            .str.upper()
        )
    elif "Country" in work_df.columns:
        work_df["_country_id_for_year_check"] = (
            work_df["Country"]
            .astype(str)
            .str.strip()
        )
    else:
        return []

    work_df["_country_id_for_year_check"] = (
        work_df["_country_id_for_year_check"]
        .replace(["", "nan", "None", "NAN", "NONE"], np.nan)
    )

    rows = []

    for column in year_columns:
        year_value = extract_year_from_column(column)

        if year_value is None:
            continue

        numeric_values = pd.to_numeric(
            work_df[column],
            errors="coerce"
        )

        available_country_count = (
            work_df.loc[
                numeric_values.notna(),
                "_country_id_for_year_check"
            ]
            .dropna()
            .nunique()
        )

        if available_country_count > 0:
            rows.append({
                "column": column,
                "year": year_value,
                "available_country_count": int(available_country_count)
            })

    counts_df = pd.DataFrame(rows)

    if counts_df.empty:
        return []

    counts_df = counts_df[
        counts_df["year"] <= current_year
    ].copy()

    if counts_df.empty:
        return []

    max_country_count = counts_df["available_country_count"].max()

    eligible_df = counts_df[
        counts_df["available_country_count"] >= max_country_count * min_share
    ].copy()

    if eligible_df.empty:
        eligible_df = counts_df.copy()

    latest_columns = (
        eligible_df
        .sort_values("year")["column"]
        .tail(2)
        .tolist()
    )

    return latest_columns

In [5]:
# ============================================================
# 4. Load template, optional manifest, and dataset file list
# ============================================================

# ----------------------------
# 4.1 Check input paths
# ----------------------------

if not os.path.exists(template_file):
    raise FileNotFoundError(
        "Template file not found: "
        + template_file
        + "\nCurrent working directory: "
        + os.getcwd()
    )

if not os.path.exists(data_folder):
    raise FileNotFoundError(
        "Data folder not found: "
        + data_folder
        + "\nRun the scraping script first."
        + "\nCurrent working directory: "
        + os.getcwd()
    )


# ----------------------------
# 4.2 Load and clean template
# ----------------------------

template_df = pd.read_csv(
    template_file,
    low_memory=False
)

template_df = normalize_columns(template_df)

# Standardize the first five expected columns if needed.
# This avoids errors caused by small naming differences.
required_template_columns = [
    "no",
    "indicator",
    "Definition",
    "unit",
    "source"
]

if len(template_df.columns) <= 5:
    raise ValueError(
        "Template file must have at least 6 columns: "
        "no, indicator, Definition, unit, source, and country columns."
    )

# Rename the first five columns to the expected names.
# This assumes your template structure is correct, even if names vary slightly.
template_columns = list(template_df.columns)
template_columns[:5] = required_template_columns
template_df.columns = template_columns

country_columns = list(template_df.columns[5:])

if len(country_columns) == 0:
    raise ValueError(
        "No country columns found in the template. "
        "The template must have country columns after the first five columns."
    )


# ----------------------------
# 4.3 Load manifest if available
# ----------------------------

if os.path.exists(manifest_file):
    manifest_df = pd.read_csv(
        manifest_file,
        low_memory=False
    )

    manifest_df = normalize_columns(manifest_df)

    print("Manifest file loaded:")
    print(manifest_file)

    print("Manifest shape:")
    print(manifest_df.shape)

else:
    manifest_df = pd.DataFrame()

    print("Warning: Manifest file was not found.")
    print("The code will continue using all dataset CSV files in the data folder.")
    print("Expected manifest path:")
    print(manifest_file)


# ----------------------------
# 4.4 List downloaded dataset CSV files
# ----------------------------

excluded_csv_names = {
    os.path.basename(manifest_file),
    os.path.basename(main_output_csv),
    os.path.basename(detail_output_csv),
    os.path.basename(rules_output_csv),
    os.path.basename(excel_output_file).replace(".xlsx", ".csv"),
    "00_Indicators_Summary.csv"
}

csv_files = sorted([
    file_name
    for file_name in os.listdir(data_folder)
    if file_name.lower().endswith(".csv")
    and file_name not in excluded_csv_names
])

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No dataset CSV files found in: "
        + data_folder
        + "\nRun the scraping script first or check the data folder path."
    )


# ----------------------------
# 4.5 Optional: keep only successful downloads from manifest
# ----------------------------

if not manifest_df.empty:
    if (
        "Downloaded_File" in manifest_df.columns
        and "Download_Status" in manifest_df.columns
    ):
        successful_files = (
            manifest_df.loc[
                manifest_df["Download_Status"].astype(str).str.lower().eq("success"),
                "Downloaded_File"
            ]
            .dropna()
            .astype(str)
            .tolist()
        )

        successful_files = [
            file_name
            for file_name in successful_files
            if file_name in csv_files
        ]

        if len(successful_files) > 0:
            csv_files = sorted(successful_files)

            print("Using successful downloads listed in manifest.")
            print("Successful dataset files used:")
            print(len(csv_files))

        else:
            print("Warning: Manifest found, but no successful downloaded files matched the folder.")
            print("The code will continue using all dataset CSV files found in the folder.")


# ----------------------------
# 4.6 Final checks and preview
# ----------------------------

print("Template shape:")
print(template_df.shape)

print("Template first five columns:")
print(list(template_df.columns[:5]))

print("Number of country columns:")
print(len(country_columns))

print("First few country columns:")
print(country_columns[:10])

print("Number of dataset files:")
print(len(csv_files))

print("First few dataset files:")
print(csv_files[:10])

Manifest file loaded:
imf_climate_country_coverage_datasets\imf_climate_country_coverage_manifest.csv
Manifest shape:
(22, 18)
Template shape:
(0, 223)
Template first five columns:
['no', 'indicator', 'Definition', 'unit', 'source']
Number of country columns:
218
First few country columns:
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba']
Number of dataset files:
23
First few dataset files:
['03_National_Greenhouse_Gas_Emissions_Inventories_and_Implied_National_Mitigation_Nationally_Determined_Contributions_Target.csv', '03_National_Greenhouse_Gas_Emissions_Inventories_and_Implied_National_Mitigation_Nationally_Determined_Contributions_Targets.csv', '04_CO2_Emissions_Emissions_Intensities_and_Emissions_Multipliers.csv', '05_CO2_Emissions_embodied_in_Domestic_Final_Demand_Production_and_Trade.csv', '06_Direct_Investment_related_Indicators.csv', '07_Environmental_Taxes.csv', '08_Environmental_Protection_Ex

In [6]:
# ============================================================
# 5. Define fields excluded from variant split rules
# ============================================================

# These columns identify reporting countries or counterpart countries.
# They are used for country matching and availability assessment,
# but they should not create separate indicator variants.
country_like_columns = {
    "Country",
    "ISO2",
    "ISO3",
    "Country_Name",
    "Country Code",
    "Country_Code",
    "Counterpart Country",
    "Counterpart_Country",
    "Counterpart ISO2",
    "Counterpart_ISO2",
    "Counterpart ISO3",
    "Counterpart_ISO3"
}

# These are metadata fields.
# They describe the dataset, source, unit, or database structure.
# They should not create separate indicator variants.
metadata_columns = country_like_columns | {
    "ObjectId",
    "OBJECTID",
    "FID",
    "Source",
    "Copyright and Usage",
    "Copyright_and_Usage",
    "CTS Code",
    "CTS_Code",
    "CTS Name",
    "CTS_Name",
    "CTS Full Descriptor",
    "CTS_Full_Descriptor",
    "Unit",
    "Scale"
}

print("Columns excluded from variant split rules:")
print(sorted(metadata_columns))

Columns excluded from variant split rules:
['CTS Code', 'CTS Full Descriptor', 'CTS Name', 'CTS_Code', 'CTS_Full_Descriptor', 'CTS_Name', 'Copyright and Usage', 'Copyright_and_Usage', 'Counterpart Country', 'Counterpart ISO2', 'Counterpart ISO3', 'Counterpart_Country', 'Counterpart_ISO2', 'Counterpart_ISO3', 'Country', 'Country Code', 'Country_Code', 'Country_Name', 'FID', 'ISO2', 'ISO3', 'OBJECTID', 'ObjectId', 'Scale', 'Source', 'Unit']


In [7]:
# ============================================================
# 6. Country name to ISO3 mapping
# ============================================================

# Manual fixes for country names that often do not map cleanly.
# These are used mainly for template country names.
manual_iso_map = {
    "Congo Dem. Rep.": "COD",
    "Congo, Dem. Rep.": "COD",
    "Democratic Republic of the Congo": "COD",
    "Congo Rep.": "COG",
    "Congo, Rep.": "COG",
    "Republic of Congo": "COG",
    "Côte dIvoire": "CIV",
    "Côte d'Ivoire": "CIV",
    "Cote dIvoire": "CIV",
    "Cote d'Ivoire": "CIV",
    "Hong Kong (S.A.R.)": "HKG",
    "Hong Kong SAR": "HKG",
    "Hong Kong SAR, China": "HKG",
    "Korea Dem. Peoples Rep.": "PRK",
    "Korea, Dem. People's Rep.": "PRK",
    "North Korea": "PRK",
    "South Korea": "KOR",
    "Korea, Rep.": "KOR",
    "Kyrgyz Republic": "KGZ",
    "Lao PDR": "LAO",
    "Lao People's Democratic Republic": "LAO",
    "Macao SAR": "MAC",
    "Macao SAR, China": "MAC",
    "Micronesia": "FSM",
    "Micronesia, Fed. Sts.": "FSM",
    "Russia": "RUS",
    "Russian Federation": "RUS",
    "São Tomé and Príncipe": "STP",
    "Sao Tome and Principe": "STP",
    "Slovak Republic": "SVK",
    "Slovakia": "SVK",
    "St. Kitts and Nevis": "KNA",
    "Saint Kitts and Nevis": "KNA",
    "St. Lucia": "LCA",
    "Saint Lucia": "LCA",
    "St. Vincent and the Grenadines": "VCT",
    "Saint Vincent and the Grenadines": "VCT",
    "Syria": "SYR",
    "Syrian Arab Republic": "SYR",
    "Taiwan Province of China": "TWN",
    "Taiwan": "TWN",
    "Türkiye": "TUR",
    "Turkey": "TUR",
    "United States": "USA",
    "United States of America": "USA",
    "Venezuela": "VEN",
    "Venezuela, RB": "VEN",
    "Vietnam": "VNM",
    "Viet Nam": "VNM",
    "Yemen": "YEM",
    "Yemen, Rep.": "YEM",
    "West Bank and Gaza": "PSE",
    "Palestine": "PSE",
    "Czech Republic": "CZE",
    "Czechia": "CZE",
    "Eswatini": "SWZ",
    "Swaziland": "SWZ",
    "Iran": "IRN",
    "Iran, Islamic Rep.": "IRN",
    "Tanzania": "TZA",
    "Tanzania, United Republic of": "TZA",
    "The Bahamas": "BHS",
    "Bahamas": "BHS",
    "Bahamas, The": "BHS",
    "The Gambia": "GMB",
    "Gambia": "GMB",
    "Gambia, The": "GMB",
    "Brunei Darussalam": "BRN",
    "Brunei": "BRN",
    "Cabo Verde": "CPV",
    "Cape Verde": "CPV",
    "Timor-Leste": "TLS",
    "Timor Leste": "TLS",
    "Channel Islands": "CHI"
}


def clean_iso3(value):
    """
    Clean ISO3 values.

    Returns None for missing or invalid-looking values.
    """
    iso_text = clean_text(value).upper()

    if iso_text in ["", "NAN", "NONE", "NULL", "NOT FOUND"]:
        return None

    if len(iso_text) != 3:
        return None

    if not iso_text.isalpha():
        return None

    return iso_text


def country_to_iso3(country_name):
    """
    Convert country name to ISO3 code.

    This first checks manual fixes, then uses country_converter.
    """
    country_clean = clean_text(country_name)

    if country_clean == "":
        return None

    # If the value is already an ISO3 code, keep it.
    possible_iso3 = clean_iso3(country_clean)
    if possible_iso3 is not None:
        return possible_iso3

    if country_clean in manual_iso_map:
        return manual_iso_map[country_clean]

    iso_value = coco.convert(
        names=country_clean,
        to="ISO3",
        not_found=None
    )

    if isinstance(iso_value, list):
        if len(iso_value) == 0:
            return None
        iso_value = iso_value[0]

    if iso_value is None:
        return None

    iso_text = clean_iso3(iso_value)

    return iso_text


template_country_map_df = pd.DataFrame({
    "country": country_columns
})

template_country_map_df["iso3"] = (
    template_country_map_df["country"]
    .map(country_to_iso3)
)

template_iso_to_country = (
    template_country_map_df
    .dropna(subset=["iso3"])
    .drop_duplicates("iso3")
    .set_index("iso3")["country"]
    .to_dict()
)

unmatched_template_countries = (
    template_country_map_df[
        template_country_map_df["iso3"].isna()
    ]["country"]
    .tolist()
)

print("Template countries mapped to ISO3:")
print(len(template_iso_to_country))

print("Total template country columns:")
print(len(country_columns))

if len(unmatched_template_countries) > 0:
    print("Warning: Some template countries were not mapped to ISO3:")
    print(unmatched_template_countries[:30])

# Save country mapping diagnostic for review.
template_country_map_output = "IMF_template_country_iso3_mapping.csv"

template_country_map_df.to_csv(
    template_country_map_output,
    index=False
)

print("Template country mapping diagnostic saved:")
print(template_country_map_output)

Template countries mapped to ISO3:
218
Total template country columns:
218
Template country mapping diagnostic saved:
IMF_template_country_iso3_mapping.csv


In [8]:
# ============================================================
# 7. Inspect split rules before processing
# ============================================================

split_rule_preview_rows = []

for file_name in tqdm(csv_files, desc="Inspecting split rules"):
    file_path = os.path.join(data_folder, file_name)

    df = pd.read_csv(file_path, low_memory=False)
    df = normalize_columns(df)

    # --------------------------------------------------------
    # 7.1 Identify year and non-year columns
    # --------------------------------------------------------

    year_columns = [
        column
        for column in df.columns
        if is_year_column(column)
    ]

    non_year_columns = [
        column
        for column in df.columns
        if column not in year_columns
    ]

    # --------------------------------------------------------
    # 7.2 Identify candidate variant split columns
    # --------------------------------------------------------

    variant_columns = [
        column
        for column in non_year_columns
        if column not in metadata_columns
    ]

    # Keep Indicator first when present.
    if "Indicator" in df.columns:
        variant_columns = (
            ["Indicator"]
            + [
                column
                for column in variant_columns
                if column != "Indicator"
            ]
        )

    # Drop fully empty variant columns.
    clean_variant_columns = []

    for column in variant_columns:
        if column in df.columns:
            nonempty_count = (
                df[column]
                .map(clean_text)
                .replace("", np.nan)
                .notna()
                .sum()
            )

            if nonempty_count > 0:
                clean_variant_columns.append(column)

    variant_columns = clean_variant_columns

    # --------------------------------------------------------
    # 7.3 Identify unit, description, and source fields
    # --------------------------------------------------------

    if "Scale" in df.columns:
        unit_source = "Scale"
    elif "Unit" in df.columns:
        unit_source = "Unit"
    else:
        unit_source = None

    if "CTS Full Descriptor" in df.columns:
        description_source = "CTS Full Descriptor"
    elif "CTS_Full_Descriptor" in df.columns:
        description_source = "CTS_Full_Descriptor"
    elif "Indicator" in df.columns:
        description_source = "Indicator"
    else:
        description_source = None

    if "Source" in df.columns:
        source_source = "Source"
    else:
        source_source = None

    # --------------------------------------------------------
    # 7.4 Inspect latest reporting columns
    # --------------------------------------------------------

    latest_columns = latest_reporting_years(
        df,
        min_share=latest_year_min_share
    )

    # --------------------------------------------------------
    # 7.5 Add preview row
    # --------------------------------------------------------

    split_rule_preview_rows.append({
        "file_name": file_name,
        "variant_columns_used": " + ".join(variant_columns),
        "unit_source": unit_source,
        "description_source": description_source,
        "source_source": source_source,
        "year_columns_found": len(year_columns),
        "first_year_column": year_columns[0] if len(year_columns) > 0 else "",
        "last_year_column": year_columns[-1] if len(year_columns) > 0 else "",
        "latest_reporting_years_used": ", ".join(latest_columns),
        "number_of_variant_columns": len(variant_columns)
    })

split_rule_preview_df = pd.DataFrame(split_rule_preview_rows)

print("Split rule preview:")
print(split_rule_preview_df.to_string(index=False))

Inspecting split rules:   0%|          | 0/23 [00:00<?, ?it/s]

Split rule preview:
                                                                                                                       file_name                                                         variant_columns_used unit_source  description_source source_source  year_columns_found first_year_column last_year_column latest_reporting_years_used  number_of_variant_columns
 03_National_Greenhouse_Gas_Emissions_Inventories_and_Implied_National_Mitigation_Nationally_Determined_Contributions_Target.csv                                              Indicator + Industry + Gas_Type       Scale CTS_Full_Descriptor        Source                  61             F1970            F2030                F2025, F2026                          3
03_National_Greenhouse_Gas_Emissions_Inventories_and_Implied_National_Mitigation_Nationally_Determined_Contributions_Targets.csv                                              Indicator + Industry + Gas_Type       Scale CTS_Full_Descriptor        Source     

In [9]:
# ============================================================
# 8. Build variant-level country availability table
# ============================================================

variant_rows = []
variant_detail_rows = []
variant_rules_rows = []
skipped_dataset_rows = []

for file_name in tqdm(csv_files, desc="Building variant availability"):
    file_path = os.path.join(data_folder, file_name)

    df = pd.read_csv(file_path, low_memory=False)
    df = normalize_columns(df)

    # --------------------------------------------------------
    # 8.1 Identify year and non-year columns
    # --------------------------------------------------------

    year_columns = [
        column
        for column in df.columns
        if is_year_column(column)
    ]

    non_year_columns = [
        column
        for column in df.columns
        if column not in year_columns
    ]

    # --------------------------------------------------------
    # 8.2 Identify variant split columns
    # --------------------------------------------------------

    variant_columns = [
        column
        for column in non_year_columns
        if column not in metadata_columns
    ]

    # Keep Indicator first when present.
    if "Indicator" in df.columns:
        variant_columns = (
            ["Indicator"]
            + [
                column
                for column in variant_columns
                if column != "Indicator"
            ]
        )

    # Drop fully empty variant columns.
    clean_variant_columns = []

    for column in variant_columns:
        if column in df.columns:
            nonempty_count = (
                df[column]
                .map(clean_text)
                .replace("", np.nan)
                .notna()
                .sum()
            )

            if nonempty_count > 0:
                clean_variant_columns.append(column)

    variant_columns = clean_variant_columns

    # --------------------------------------------------------
    # 8.3 Identify unit, description, and source columns
    # --------------------------------------------------------

    if "Scale" in df.columns:
        unit_column = "Scale"
    elif "Unit" in df.columns:
        unit_column = "Unit"
    else:
        unit_column = None

    if "CTS Full Descriptor" in df.columns:
        description_column = "CTS Full Descriptor"
    elif "CTS_Full_Descriptor" in df.columns:
        description_column = "CTS_Full_Descriptor"
    elif "Indicator" in df.columns:
        description_column = "Indicator"
    else:
        description_column = None

    if "Source" in df.columns:
        source_column = "Source"
    else:
        source_column = None

    # --------------------------------------------------------
    # 8.4 Select latest reporting columns
    # --------------------------------------------------------

    latest_columns = latest_reporting_years(
        df,
        min_share=latest_year_min_share
    )

    indicator_id_match = re.match(r"^(\d+)_", file_name)

    if indicator_id_match is not None:
        indicator_id = int(indicator_id_match.group(1))
    else:
        indicator_id = len(variant_rules_rows) + 1

    variant_rules_rows.append({
        "file_name": file_name,
        "variant_columns_used": " + ".join(variant_columns),
        "unit_source": unit_column,
        "description_source": description_column,
        "source_source": source_column if source_column is not None else "",
        "latest_reporting_years_used": ", ".join(latest_columns),
        "year_columns_found": len(year_columns)
    })

    if len(latest_columns) == 0:
        skipped_dataset_rows.append({
            "file_name": file_name,
            "reason": "No valid latest reporting year columns found"
        })
        continue

    valid_latest_columns = [
        column
        for column in latest_columns
        if column in df.columns
    ]

    if len(valid_latest_columns) == 0:
        skipped_dataset_rows.append({
            "file_name": file_name,
            "reason": "Latest reporting year columns not found in dataframe"
        })
        continue

    # --------------------------------------------------------
    # 8.5 Define columns to keep
    # --------------------------------------------------------

    keep_columns = []

    if "Country" in df.columns:
        keep_columns.append("Country")

    if "ISO3" in df.columns:
        keep_columns.append("ISO3")

    if len(keep_columns) == 0:
        skipped_dataset_rows.append({
            "file_name": file_name,
            "reason": "No Country or ISO3 column found"
        })
        continue

    for column in variant_columns:
        if column in df.columns and column not in keep_columns:
            keep_columns.append(column)

    for extra_column in [description_column, unit_column, source_column]:
        if extra_column is not None:
            if extra_column in df.columns and extra_column not in keep_columns:
                keep_columns.append(extra_column)

    work_df = df[
        keep_columns + valid_latest_columns
    ].copy()

    # --------------------------------------------------------
    # 8.6 Mark rows with data in at least one selected latest year
    # --------------------------------------------------------

    work_df["has_latest_data"] = (
        work_df[valid_latest_columns]
        .apply(pd.to_numeric, errors="coerce")
        .notna()
        .any(axis=1)
    )

    work_df = work_df.loc[
        work_df["has_latest_data"]
    ].copy()

    if work_df.empty:
        skipped_dataset_rows.append({
            "file_name": file_name,
            "reason": "No rows had data in the selected latest reporting years"
        })
        continue

    # --------------------------------------------------------
    # 8.7 Match reporting country to template country
    # --------------------------------------------------------

    # Country availability uses the reporting country, not counterpart country.
    if "ISO3" in work_df.columns:
        work_df["matched_iso3"] = (
            work_df["ISO3"]
            .map(clean_iso3)
        )

    elif "Country" in work_df.columns:
        work_df["matched_iso3"] = (
            work_df["Country"]
            .map(country_to_iso3)
        )

    else:
        work_df["matched_iso3"] = None

    work_df = work_df.loc[
        work_df["matched_iso3"].isin(template_iso_to_country.keys())
    ].copy()

    if work_df.empty:
        skipped_dataset_rows.append({
            "file_name": file_name,
            "reason": "No reporting countries matched template countries"
        })
        continue

    # --------------------------------------------------------
    # 8.8 Build variant indicator label
    # --------------------------------------------------------

    for column in variant_columns:
        if column in work_df.columns:
            work_df[column] = work_df[column].map(clean_text)

    if len(variant_columns) > 0:
        work_df["variant_indicator"] = (
            work_df[variant_columns]
            .astype(str)
            .agg(" | ".join, axis=1)
        )

        work_df["variant_indicator"] = (
            work_df["variant_indicator"]
            .map(clean_text)
        )

    else:
        work_df["variant_indicator"] = "Dataset " + str(indicator_id)

    # --------------------------------------------------------
    # 8.9 Create one output row per indicator variant
    # --------------------------------------------------------

    variant_number = 0

    for variant_key, group_df in work_df.groupby(
        "variant_indicator",
        dropna=False
    ):
        variant_number = variant_number + 1

        output_row = {
            column: 0
            for column in template_df.columns
        }

        output_row["no"] = (
            str(indicator_id).zfill(2)
            + "-"
            + str(variant_number).zfill(4)
        )

        output_row["indicator"] = variant_key

        if (
            description_column is not None
            and description_column in group_df.columns
        ):
            output_row["Definition"] = first_nonempty(
                group_df[description_column]
            )
        else:
            output_row["Definition"] = variant_key

        if (
            unit_column is not None
            and unit_column in group_df.columns
        ):
            output_row["unit"] = first_nonempty(
                group_df[unit_column]
            )
        else:
            output_row["unit"] = ""

        if (
            source_column is not None
            and source_column in group_df.columns
        ):
            output_row["source"] = unique_join(
                group_df[source_column]
            )
        else:
            output_row["source"] = "IMF Climate Data"

        if output_row["source"] == "":
            output_row["source"] = "IMF Climate Data"

        # Mark countries with available data.
        available_iso3_values = set(
            group_df["matched_iso3"]
            .dropna()
            .astype(str)
        )

        for iso3_value in available_iso3_values:
            country_name = template_iso_to_country.get(iso3_value)

            if country_name in output_row:
                output_row[country_name] = 1

        available_country_count = int(
            sum(
                output_row[country_name]
                for country_name in country_columns
            )
        )

        variant_rows.append(output_row)

        variant_detail_rows.append({
            "no": output_row["no"],
            "dataset_no": indicator_id,
            "file_name": file_name,
            "indicator": output_row["indicator"],
            "Definition": output_row["Definition"],
            "unit": output_row["unit"],
            "source": output_row["source"],
            "variant_columns_used": " + ".join(variant_columns),
            "latest_reporting_years_used": ", ".join(valid_latest_columns),
            "available_country_count": available_country_count
        })

Building variant availability:   0%|          | 0/23 [00:00<?, ?it/s]

In [10]:
# ============================================================
# 9. Create final DataFrames
# ============================================================

final_availability_df = pd.DataFrame(
    variant_rows,
    columns=template_df.columns
)

if not final_availability_df.empty:
    final_availability_df = (
        final_availability_df
        .sort_values("no")
        .reset_index(drop=True)
    )

variant_detail_df = pd.DataFrame(variant_detail_rows)

if not variant_detail_df.empty:
    variant_detail_df = (
        variant_detail_df
        .sort_values("no")
        .reset_index(drop=True)
    )

split_rules_df = pd.DataFrame(variant_rules_rows)

skipped_datasets_df = pd.DataFrame(skipped_dataset_rows)

print("Final availability shape:")
print(final_availability_df.shape)

print("Variant detail shape:")
print(variant_detail_df.shape)

print("Split rules shape:")
print(split_rules_df.shape)

print("Skipped datasets shape:")
print(skipped_datasets_df.shape)

print("Preview:")
print(final_availability_df.head())

Final availability shape:
(1089, 223)
Variant detail shape:
(1089, 10)
Split rules shape:
(23, 7)
Skipped datasets shape:
(0, 0)
Preview:
        no                                          indicator  \
0  03-0001  IMF estimated CO2 emissions under a Business a...   
1  03-0001  IMF estimated CO2 emissions under a Business a...   
2  03-0002  IMF estimated GHG emissions excluding land-use...   
3  03-0002  IMF estimated GHG emissions excluding land-use...   
4  03-0003  IMF estimated GHG emissions including land-use...   

                                          Definition   unit  \
0  Environment, Climate Change, Greenhouse Gas Em...  Units   
1  Environment, Climate Change, Greenhouse Gas Em...  Units   
2  Environment, Climate Change, Greenhouse Gas Em...  Units   
3  Environment, Climate Change, Greenhouse Gas Em...  Units   
4  Environment, Climate Change, Greenhouse Gas Em...  Units   

                                              source  Afghanistan  Albania  \
0  United Nati

In [11]:
# ============================================================
# 10. Validate important split-rule fixes
# ============================================================

important_rules_df = split_rules_df[
    split_rules_df["file_name"].str.startswith("12_")
    | split_rules_df["file_name"].str.startswith("19_")
].copy()

print("Important split-rule checks:")
print(important_rules_df.to_string(index=False))

print("")
print("Expected checks:")
print("Dataset 12 should use: Indicator + Trade Flow")
print("Dataset 19 should use: Indicator + Growth Type + Sector + Scenarios + Scope")
print("Country, Counterpart Country, ISO fields, and Copyright fields should not appear.")

Important split-rule checks:
                                               file_name                                 variant_columns_used unit_source  description_source source_source latest_reporting_years_used  year_columns_found
12_Bilateral_Trade_in_Low_Carbon_Technology_Products.csv                               Indicator + Trade_Flow       Scale CTS_Full_Descriptor        Source                F2023, F2024                  31
19_Carbon_Tax_to_Revenues_Assets_in_Disclosing_Firms.csv Indicator + Growth_Type + Sector + Scenarios + Scope        Unit CTS_Full_Descriptor        Source                F2025, F2026                  26

Expected checks:
Dataset 12 should use: Indicator + Trade Flow
Dataset 19 should use: Indicator + Growth Type + Sector + Scenarios + Scope
Country, Counterpart Country, ISO fields, and Copyright fields should not appear.


In [12]:
# ============================================================
# 11. Save CSV outputs
# ============================================================

final_availability_df.to_csv(
    main_output_csv,
    index=False
)

variant_detail_df.to_csv(
    detail_output_csv,
    index=False
)

split_rules_df.to_csv(
    rules_output_csv,
    index=False
)

skipped_output_csv = "IMF_final_variant_skipped_datasets.csv"

skipped_datasets_df.to_csv(
    skipped_output_csv,
    index=False
)

print("Saved CSV files:")
print(main_output_csv)
print(detail_output_csv)
print(rules_output_csv)
print(skipped_output_csv)

Saved CSV files:
IMF_final_variant_country_availability.csv
IMF_final_variant_country_availability_detail.csv
IMF_final_variant_split_rules.csv
IMF_final_variant_skipped_datasets.csv


In [13]:
# ============================================================
# 12. Save Excel workbook with all sheets
# ============================================================

with pd.ExcelWriter(excel_output_file, engine="xlsxwriter") as writer:
    final_availability_df.to_excel(
        writer,
        sheet_name="country_availability",
        index=False
    )

    variant_detail_df.to_excel(
        writer,
        sheet_name="variant_details",
        index=False
    )

    split_rules_df.to_excel(
        writer,
        sheet_name="split_rules",
        index=False
    )

    skipped_datasets_df.to_excel(
        writer,
        sheet_name="skipped_datasets",
        index=False
    )

    split_rule_preview_df.to_excel(
        writer,
        sheet_name="split_rule_preview",
        index=False
    )

print("Saved Excel file:")
print(excel_output_file)

Saved Excel file:
IMF_final_variant_country_availability.xlsx


In [14]:
# ============================================================
# 13. Final validation summary
# ============================================================

print("Final validation summary")
print("------------------------")

print("Final row count:")
print(len(final_availability_df))

print("Final column count:")
print(len(final_availability_df.columns))

print("Detail row count:")
print(len(variant_detail_df))

print("Rules row count:")
print(len(split_rules_df))

print("Skipped dataset count:")
print(len(skipped_datasets_df))

print("Main CSV file exists:")
print(os.path.exists(main_output_csv))

print("Excel file exists:")
print(os.path.exists(excel_output_file))

if os.path.exists(main_output_csv):
    print("Main CSV file size in bytes:")
    print(os.path.getsize(main_output_csv))

if os.path.exists(excel_output_file):
    print("Excel file size in bytes:")
    print(os.path.getsize(excel_output_file))

print("Files created successfully.")

Final validation summary
------------------------
Final row count:
1089
Final column count:
223
Detail row count:
1089
Rules row count:
23
Skipped dataset count:
0
Main CSV file exists:
True
Excel file exists:
True
Main CSV file size in bytes:
825191
Excel file size in bytes:
643031
Files created successfully.
